# NB03 — Myocardium Signal Metrics

**Physics rationale:** Off-resonance reduces steady-state bSSFP signal. At the optimal frequency offset,
the myocardium signal is maximised and most homogeneous.

**Metrics:** `mu_myo`, `snr_myo`, `sigma_myo`, `homogeneity` (1 − CoV)

**Selector:** `argmax(mu_myo)` vs GT

**Runs on:** Local CPU

## 0. Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

Dependencies OK


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE    = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]
print('Config OK')

Config OK


## 1. Compute Signal Metrics

In [3]:
def background_pixels(gray):
    H, W = gray.shape
    c = max(10, H // 8)
    return np.concatenate([
        gray[:c, :c].ravel(), gray[:c, -c:].ravel(),
        gray[-c:, :c].ravel(), gray[-c:, -c:].ravel()
    ])

def align_mask(mask, target_shape):
    th, tw = target_shape[:2]
    mh, mw = mask.shape[:2]
    if (mh, mw) == (th, tw): return mask.astype(bool)
    if (mh, mw) == (tw, th): return mask.T.astype(bool)
    from PIL import Image as PILImage
    pil = PILImage.fromarray(mask.astype(np.uint8) * 255)
    pil = pil.resize((tw, th), PILImage.NEAREST)
    return (np.asarray(pil) > 127)

records = []
for pid in sorted(all_data):
    if pid in EXCLUDE: continue
    data = all_data[pid]
    seg  = seg_cache.get(pid)
    if seg is None: continue

    for i, (img, hz) in enumerate(zip(data['images'], data['offsets_hz'])):
        gray = img.astype(np.float32)
        if gray.max() > 1.0: gray /= gray.max()
        myo_mask = align_mask(seg.myocardium_mask, gray.shape)
        myo_px   = gray[myo_mask]
        bg_px    = background_pixels(gray)

        mu_myo    = float(myo_px.mean()) if myo_px.size > 0 else 0.0
        sigma_myo = float(myo_px.std())  if myo_px.size > 0 else 1e-6
        sigma_bg  = max(float(bg_px.std()), 1e-6)
        cov       = sigma_myo / max(mu_myo, 1e-6)
        snr_myo   = mu_myo / sigma_bg
        homog     = max(0.0, 1.0 - cov)

        records.append({
            'patient_id': pid, 'image_no': i + 1, 'offset_hz': hz,
            'mu_myo': round(mu_myo, 6), 'sigma_myo': round(sigma_myo, 6),
            'snr_myo': round(snr_myo, 4), 'homogeneity': round(homog, 4), 'cov': round(cov, 4),
        })

df = pd.DataFrame(records)
df.to_csv(f'{OUT_DIR}/signal_metrics.csv', index=False)
print(f'Computed {len(df)} rows. Saved signal_metrics.csv')
df.head(13)

NameError: name 'all_data' is not defined

## 2. Metric Curves per Patient

In [ ]:
from utils.vis_utils import plot_metric_curves
import matplotlib.pyplot as plt

for metric_col, title in [
    ('mu_myo',     'Myocardium Mean Signal (mu_myo)'),
    ('snr_myo',    'Myocardium SNR'),
    ('homogeneity','Signal Homogeneity (1 - CoV)'),
]:
    metric_dict = {
        pid: df[df['patient_id']==pid].sort_values('image_no')[metric_col].tolist()
        for pid in sorted(df['patient_id'].unique())
    }
    fig = plot_metric_curves(
        metric_dict=metric_dict, offsets_hz=OFFSETS_13, gt_dict=gt_dict,
        title=title, ylabel=metric_col,
        save_path=f'{OUT_DIR}/signal_{metric_col}_curves.png',
    )
    plt.show()

## 3. Heatmap — mu_myo across all patients × offsets

In [ ]:
from utils.vis_utils import plot_heatmap
import matplotlib.pyplot as plt

patient_ids_sorted = sorted(df['patient_id'].unique())
matrix = np.zeros((len(patient_ids_sorted), 13))
for row_i, pid in enumerate(patient_ids_sorted):
    sub = df[df['patient_id']==pid].sort_values('image_no')
    vals = sub['mu_myo'].values
    matrix[row_i, :len(vals)] = vals

fig = plot_heatmap(
    data_matrix=matrix, patient_ids=patient_ids_sorted, offsets_hz=OFFSETS_13,
    gt_dict=gt_dict, title='Myocardium Mean Signal — GT box = green',
    cbar_label='mu_myo', cmap='plasma',
    save_path=f'{OUT_DIR}/signal_mu_myo_heatmap.png',
)
plt.show()

## 4. Accuracy Evaluation

In [ ]:
from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

for selector_col, selector_name, argfn in [
    ('mu_myo',     'argmax(mu_myo)',      np.argmax),
    ('snr_myo',    'argmax(snr_myo)',     np.argmax),
    ('homogeneity','argmax(homogeneity)', np.argmax),
]:
    pred_dict = {}
    for pid in df['patient_id'].unique():
        sub = df[df['patient_id']==pid].sort_values('image_no')
        pred_dict[pid] = int(argfn(sub[selector_col].values)) + 1

    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    print(summary_text(selector_name, res['top1'], res['pm1'], res['mae_hz'], res['n_valid']))

print('NB03 complete. Proceed to nb04_contrast_metrics.ipynb')